# Scan2Stage — Colab operator notebook

Upload a textured UGScan/GLB scan, run the detector, and immediately inspect a semantic top-view map. The map uses lines and symbols instead of raw candidate dots.


## 1. Install / update Scan2Stage


In [ ]:
REPO_URL = 'https://github.com/6564200/Scan2Stage.git'
WORKDIR = '/content/Scan2Stage'

!rm -rf {WORKDIR}
!git clone --depth 1 --branch main {REPO_URL} {WORKDIR}
%cd {WORKDIR}
!bash scripts/colab_bootstrap.sh

import scan2stage
print('Scan2Stage ready:', scan2stage.__version__)


## 2. Upload scan
Supported here: `.glb`, `.gltf`, `.zip`, `.fbx`, `.obj`.


In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
name = next(iter(uploaded))
allowed = {'.glb', '.gltf', '.zip', '.fbx', '.obj'}
assert Path(name).suffix.lower() in allowed, f'Unsupported file: {name}'

INPUT = Path('/content/Scan2Stage/data') / Path(name).name
INPUT.parent.mkdir(parents=True, exist_ok=True)
Path(name).replace(INPUT)
print('Input:', INPUT)


## 3. Run detector
For a first pass use 300k samples. Increase to 500k–750k only after checking speed and recall.


In [ ]:
import subprocess, time, shutil

SAMPLES = 300_000
SOURCE_UP = 'y'
OUT = Path('/content/Scan2Stage/outputs') / INPUT.stem
if OUT.exists():
    shutil.rmtree(OUT)

cmd = [
    'scan2stage', str(INPUT),
    '--output-dir', str(OUT),
    '--samples', str(SAMPLES),
    '--source-up', SOURCE_UP,
]
print('Running:', ' '.join(cmd))
t0 = time.time()
subprocess.run(cmd, check=True)
print(f'Finished in {time.time() - t0:.1f} s')
print('Output:', OUT)


## 4. Result summary


In [ ]:
import json
import pandas as pd
from IPython.display import display
from scan2stage.visualize import object_summary

scene = json.loads((OUT / 'structural_scene.json').read_text())
print('Counts:')
display(pd.DataFrame([scene['counts']]))

print('Shooting direction:')
display(pd.DataFrame([scene['shooting_direction']]))

rows = object_summary(scene)
objects_df = pd.DataFrame(rows)
display(objects_df)


## 5. Semantic top-view map

**Legend:** walls/partitions are gray lines; large structural/trap candidates brown; compact decor orange; Fault Lines red; cardboard target hypotheses green with an arrow toward the athlete; metal targets blue with an icon. Future Popper/Plate subtype labels automatically use their own icons.


In [ ]:
from scan2stage.visualize import render_semantic_topview

MAP = OUT / 'semantic_topview.png'
fig, ax, scene = render_semantic_topview(
    OUT / 'structural_scene.json',
    OUT / 'topview_layers.npz',
    output_path=MAP,
    show_labels=True,
    show_density=True,
)
print('Saved:', MAP)


## 6. Optional debug view
Inspect raw height layers when a target or decoration looks wrong.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

layers = np.load(OUT / 'topview_layers.npz')
slice_names = sorted([k for k in layers.files if k.startswith('slice_')])
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, key, height_range in zip(axes.flat, slice_names, scene['topview']['height_slice_ranges_m']):
    ax.imshow(np.log1p(layers[key]), origin='lower', cmap='gray')
    ax.set_title(f'{height_range[0]:.2f}–{height_range[1]:.2f} m')
    ax.axis('off')
plt.tight_layout()
plt.show()


## 7. Download results


In [ ]:
archive = shutil.make_archive(str(OUT), 'zip', OUT)
print('Archive:', archive)
files.download(archive)


## 8. Optional: run repository tests


In [ ]:
!pytest -q
